# Proteome exploration with ESMC embeddings

A **single** scanpy graph drives everything: a KNN graph on the ESMC embeddings, one UMAP layout, and Leiden clusters — all from the same neighbors graph, with a fixed seed for reproducibility. SAE features are tested for enrichment per cluster (proteins as "cells", SAE features as "genes") and annotated with descriptions from the ESM Atlas.

**Prerequisites:** run the embedding step first, and `pip install -e "..[cluster]"` (scanpy, leidenalg, igraph). Needs `BASEROW_TOKEN` / `BIOHUB_API_TOKEN` in the env.

In [ ]:
from och_annotate.config import load_config
from och_annotate.analysis import load_embeddings

cfg = load_config("../config/octopus_chierchiae.yaml")
df = load_embeddings(cfg, prefer_cache=True)   # prefer_cache=False to pull from Baserow
print(f"Loaded {len(df)} embeddings; vector dim = {len(df['embedding'].iloc[0])}")
df.head()

## One graph: KNN → UMAP → Leiden

Defaults match the original UMAP (`n_neighbors=15`, `min_dist=0.1`, cosine). Tune **granularity** here: `N_NEIGHBORS`/`MIN_DIST` shape the UMAP, `LEIDEN_RES` sets cluster count (higher = more, finer clusters).

In [ ]:
import scanpy as sc
from och_annotate.analysis import build_anndata, sae_enrichment, plot_umap

# ---- tunable parameters (defaults reproduce the original UMAP) ----
SEED        = 0
N_NEIGHBORS = 15        # KNN/UMAP granularity (local <-> global structure)
MIN_DIST    = 0.1       # UMAP point spread
METRIC      = "cosine"  # suits language-model embeddings
LEIDEN_RES  = 1.0       # clustering granularity (higher = more clusters)

# Build one AnnData and one neighbors graph; UMAP and Leiden both use it.
adata = build_anndata(df)
sc.pp.neighbors(adata, use_rep="X_esmc", n_neighbors=N_NEIGHBORS, metric=METRIC, random_state=SEED)
sc.tl.umap(adata, min_dist=MIN_DIST, random_state=SEED)
sc.tl.leiden(adata, resolution=LEIDEN_RES, flavor="igraph", n_iterations=2,
             directed=False, random_state=SEED)

# Single source of UMAP coords + cluster labels for every plot below.
meta_cols = [c for c in df.columns if c not in ("embedding", "sae_top_features")]
coords = df[meta_cols].copy().reset_index(drop=True)
coords["umap_0"] = adata.obsm["X_umap"][:, 0]
coords["umap_1"] = adata.obsm["X_umap"][:, 1]
coords["leiden"] = adata.obs["leiden"].to_numpy()
print(f"{adata.n_obs} proteins; {coords['leiden'].nunique()} Leiden clusters")

In [ ]:
# Interactive UMAP (scanpy graph) colored by chromosome
plot_umap(coords, color="chromosome", title=f"{cfg.name} — UMAP (chromosome)").show()

In [ ]:
# Same UMAP, colored by Leiden cluster
plot_umap(coords, color="leiden", title=f"{cfg.name} — UMAP (Leiden clusters)").show()

## SAE-feature enrichment per cluster

Wilcoxon rank-sum on the SAE activation matrix — the marker-gene test with SAE features in place of genes. Enriched features are annotated with their **ESM Atlas** `label`/`category`.

In [ ]:
from och_annotate.atlas import fetch_feature_descriptions

# Per-cluster enriched SAE features (FDR in pvals_adj)
enrich = sae_enrichment(adata, groupby="leiden", method="wilcoxon", n=15)

# Annotate with ESM Atlas descriptions (only the features shown; cached; no Biohub credits)
feat_ids = sorted(enrich["sae_feature"].astype(int).unique())
desc = fetch_feature_descriptions(feat_ids, cache_path="../data/sae_feature_descriptions.parquet")
lab = dict(zip(desc["feature"].astype(str), desc["label"]))
catg = dict(zip(desc["feature"].astype(str), desc["category"]))
enrich["label"] = enrich["sae_feature"].astype(str).map(lab)
enrich["category"] = enrich["sae_feature"].astype(str).map(catg)
print(f"Annotated {len(desc)} features from the ESM Atlas")
enrich[["leiden", "sae_feature", "label", "category", "scores", "pvals_adj"]].head(30)

In [ ]:
# Top-5 enriched SAE features per cluster, with ESM Atlas labels
top5 = (enrich.sort_values(["leiden", "scores"], ascending=[True, False])
              .groupby("leiden", observed=True).head(5))
for cl, g in top5.groupby("leiden", observed=True):
    print(f"\ncluster {cl}:")
    for r in g.itertuples():
        print(f"  [{r.sae_feature}] {r.label}  ({r.category})  score={r.scores:.1f}")

# Dotplot of marker SAE features across clusters
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

### SAE feature descriptions (ESM Atlas)

Descriptions come from the public **ESM Atlas** API
(`biohub.ai/esm/protein/api/v1alpha1/features/{idx}`) via
`och_annotate.atlas.fetch_feature_descriptions` — it pulls the features in the
enrichment table (concurrent, cached under `data/`, and **not** charged against
Biohub embedding credits) and adds `label`/`summary`/`category`. Browse features
at https://biohub.ai/esm/protein/atlas .

### Other next steps
- Write `adata.obs["leiden"]` back to Baserow as a `leiden_cluster` column.
- Tune `LEIDEN_RES` (cluster granularity) and `N_NEIGHBORS` / `MIN_DIST` (UMAP).
- Enrichment sharpens as SAE coverage completes across the proteome.